# 🧠 Practical Generative AI Workshop
### From Understanding AI Concepts → Building a Real AI System
**Stack:** Python + LangChain + Google Gemini · **Runs in:** Google Colab

---

**Where we left off:** you already know GenAI fundamentals, the different ways to use LLMs, model types, prompt engineering, and core AI terminology.

**Where we're going today:** we stop *talking about* LLMs and start *building with* one — step by step, until we have a small but real AI application (chatbot + memory + retrieval over a document).

> 👋 **Not a Python expert? That's fine.** This notebook is about AI concepts and system design, not Python mastery. Every code cell is short, and the "what/why" is always explained before and after. The concepts you learn (frameworks, memory, embeddings, RAG) apply the same way whether your team later builds this in Python, Java, JS/TS, or C#.

### Today's roadmap
1. Introduction to AI Frameworks
2. Configure the LLM (Gemini via LangChain)
3. LLM Parameters (temperature, max tokens, model choice)
4. Build a Basic Chatbot
5. Conversation History & Memory
6. Introduce RAG (Retrieval-Augmented Generation)
7. Build a Small RAG System
8. Final Mini AI System — put it all together

## 1. Introduction to AI Frameworks

**What is an AI/LLM framework?**
Underneath every "AI feature" is really just an HTTP call to a model: you send text in, you get text back. A **framework** is a library that sits on top of that raw API call and gives you reusable building blocks for the patterns you'll use *every single time* — sending prompts, keeping conversation state, chunking and searching documents, calling tools/functions, chaining multiple steps together.

**Why bother with a framework instead of calling the raw API yourself?**
- Less boilerplate — prompt templates, retries, output parsing already done for you.
- Swappable models — same code structure works whether the model behind it is Gemini, GPT, Claude, or a local model.
- Built-in patterns for the exact things we're about to build: memory, retrieval (RAG), agents/tools.
- Large community, docs, and pre-built integrations (vector databases, document loaders, etc).

**Popular frameworks (this concept exists in every language):**

| Framework | Primary language(s) | Notes |
|---|---|---|
| **LangChain** | Python, JavaScript/TypeScript | Most widely adopted, general-purpose orchestration |
| **LlamaIndex** | Python, TypeScript | Strong focus on data/RAG pipelines |
| **Semantic Kernel** | C#, Python, Java | Microsoft's framework, popular in .NET shops |
| **LangChain4j / Spring AI** | Java | LangChain-style concepts for the JVM |
| **Vercel AI SDK** | JavaScript/TypeScript | Popular for web app front-ends |
| Provider SDKs (`google-generativeai`, `openai`, `anthropic`) | Every major language | Lower-level, no orchestration built in |

**Where does LangChain fit?** It's a general-purpose orchestration layer: prompt templates → memory → retrieval → tools/agents, all behind one consistent interface, with adapters for almost every LLM provider (including Gemini, which we'll use today).

**Why Python + LangChain for this workshop?**
- Python is the de-facto language of the AI/ML ecosystem — most tutorials, papers, and new integrations show up here first.
- LangChain is the most mature and most documented of these frameworks, so it's the fastest way to *see the concepts in action*.
- Zero local setup — this entire notebook runs in Google Colab in a browser.
- Every concept you learn today (frameworks, memory, embeddings, RAG) maps 1:1 onto whatever language/framework your own team uses — only the syntax changes.

We're deliberately keeping the LangChain usage **minimal**. The goal is that you understand *what a chatbot/RAG system is actually doing*, not that you memorize a framework's API surface.


## 2. Configure the LLM

We'll now go from zero to "talking to Gemini" in four steps:
1. Install the required packages.
2. Provide your Gemini API key.
3. Initialize the Gemini model through LangChain.
4. Make our first LLM call.

### 2.1 Install packages


In [ ]:
# LangChain core + the Gemini integration + Google's own SDK (used for a couple of direct calls later)
!pip install -q langchain langchain-core langchain-google-genai google-generativeai langchain-text-splitters


`langchain-core` gives us the framework's core building blocks (messages, prompts, vector store interfaces). `langchain-google-genai` is the adapter that lets LangChain talk to Gemini. `google-generativeai` is Google's own SDK — LangChain uses it under the hood, and we'll use it directly once to *list available models*.

### 2.2 Provide your Gemini API key (securely)

Never write an API key directly into a notebook cell — if you share the notebook or push it to a repo, the key leaks with it. Instead we'll use `getpass`, which prompts for input **without echoing it to the screen**, and store it only in this runtime's memory (as an environment variable) — it disappears when the Colab runtime is reset.

> Don't have a key yet? Get one for free at **[Google AI Studio](https://aistudio.google.com/app/apikey)**.


In [ ]:
import os
from getpass import getpass

# You will be prompted to paste your key below. It will NOT be displayed or saved to the notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("API key loaded into this session ✅ (not printed, not saved to the notebook file)")


Enter your Gemini API key: ··········
API key loaded into this session ✅ (not printed, not saved to the notebook file)


### 2.3 Initialize the Gemini model through LangChain

`ChatGoogleGenerativeAI` is a LangChain **wrapper class** around Gemini's chat API. Once wrapped, it behaves the same way every other LangChain chat model does — this is exactly the "swappable model" benefit we mentioned in Section 1.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # a fast, low-cost Gemini model — good default for a workshop
    temperature=0.7,                # we'll explain this in Section 3
)

print("Model initialized:", llm.model)


Model initialized: gemini-3.1-flash-lite


### 2.4 Your first LLM call


In [ ]:
response = llm.invoke("In one sentence, explain what a Large Language Model is.")
print(response.content)


[{'type': 'text', 'text': 'A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.', 'extras': {'signature': 'EnEKbwERTTIPsSgIwJ7GPtoCy56Beat73BPXZi+U+hXJBOTIZWqT3SqIRE2vR2MI8Byi5GtEmJJnvN3AVnND8I7jYpVqsxrRM+RM7hiz83YgddYi1aKItCSRZtblFufJM0jnD1znmk49kSMf7/zWiNqbqw=='}}]


**What just happened behind the scenes?**

1. `llm.invoke("...")` took your plain string and wrapped it into a request Gemini understands.
2. LangChain sent that request over HTTPS to Google's Gemini endpoint, authenticated using your API key.
3. Gemini generated a reply token-by-token on Google's servers.
4. The full reply came back as an `AIMessage` object — `response`.

**One quirk to flag before we move on:** `response.content` is *usually* a plain string — but depending on the model, it can also come back as a **list of content blocks** instead. Look at what the cell above printed: if it was a plain sentence, you got a string; if it looked like `[{'type': 'text', 'text': '...', 'extras': {...}}]`, you got the list form (some Gemini 3 models attach an internal "thought signature" block alongside the visible text — that's the `extras` part, and we don't care about it).

Either way, all we ever want is the visible text. So instead of writing `.content` everywhere and handling both cases every time, let's write one tiny helper *once* and reuse it for every response for the rest of the workshop — that keeps the comparisons in Section 3 (and everything after) easy to read.


In [ ]:
def get_text(response):
    """Return just the plain text of an LLM response, whether `.content` is a string
    or a list of content blocks (e.g. text + an internal thought-signature block)."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

print(get_text(response))


A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.


**Important:** that call was completely self-contained. Gemini does not remember you asked it anything — it has no memory between calls. Keep that in mind; it's the whole reason Section 5 exists.


## 3. LLM Parameters

Every LLM call is controlled by a handful of parameters. The three that matter most day-to-day:

| Parameter | What it controls | Typical range |
|---|---|---|
| `temperature` | Randomness / "creativity" of the output | `0.0` (deterministic) → `1.0`+ (more random) |
| `max_output_tokens` | Hard cap on how long the response can be | depends on model, e.g. 1–8192 |
| `model` | Which underlying model answers — trades off speed/cost vs. capability | e.g. `gemini-3.1-flash-lite` vs `gemini-3.5-flash` |

Let's *see* the effect of each one instead of just reading about it.

### 3.1 Temperature: low vs. high, same prompt


In [ ]:
prompt = "Give me a one-sentence tagline for a coffee shop."

llm_low_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)
llm_high_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=1.0)

print("=== temperature = 0.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_low_temp.invoke(prompt)))

print("\n=== temperature = 1.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_high_temp.invoke(prompt)))


=== temperature = 0.0 (run 3 times) ===
1. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
2. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
3. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*

**Expected behavior:** the `temperature=0.0` runs should come back nearly identical (or exactly identical) each time — the model is picking the single most-likely next word every time. The `temperature=1.0` runs should vary more from each other — the model is now sometimes picking less-likely (but still plausible) words, trading consistency for variety.

**When to use which:** low temperature for factual/deterministic tasks (data extraction, classification, code generation), higher temperature for creative tasks (brainstorming, marketing copy, varied phrasing).

### 3.2 Maximum output tokens

Tokens are roughly "chunks of text" (not quite words, not quite characters). `max_output_tokens` puts a hard ceiling on the length of the response — useful for controlling cost/latency, or for forcing short answers.


In [ ]:
prompt = "Explain how a vector database works."

llm_short = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=20)
llm_long = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=300)

print("=== max_output_tokens = 20 ===")
print(get_text(llm_short.invoke(prompt)))

print("\n=== max_output_tokens = 300 ===")
print(get_text(llm_long.invoke(prompt)))


=== max_output_tokens = 20 ===
To understand how a vector database works, you first have to understand the concept of

=== max_output_tokens = 300 ===
To understand a vector database, you first have to understand **Vector Embeddings**.

### 1. The Foundation: What is a Vector?
In traditional databases (SQL), data is stored in rows and columns. You search by exact matches (e.g., "Find the user with ID 123").

In AI, data (text, images, audio) is converted into **Vector Embeddings**. An embedding is simply a long list of numbers (a coordinate) that represents the **meaning** of the data. 

For example, the word "King" might be represented as `[0.9, 0.1, 0.8]`. If you have a model that understands language, it will place "King" and "Queen" very close to each other in a multi-dimensional space, while "Apple" would be placed far away.

### 2. How a Vector Database Works
A vector database is designed specifically to store these lists of numbers and perform "similarity searches" across them. 

**Expected behavior:** the first response should be cut off mid-thought — it hit the token ceiling before finishing. The second should be a complete explanation. This is why real applications set a sensible `max_output_tokens`: an unbounded response can be slow and expensive, but too small a cap gives truncated, unusable answers.

### 3.3 Model selection

Gemini (like most providers) offers several model sizes — smaller/faster/cheaper vs. larger/slower/more capable. Which one is "current" changes over time, so rather than trusting a hardcoded name, let's ask the API what's actually available right now.


In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Models available to your key that support chat:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(" -", m.name)


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Models available to your key that support chat:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - mode

We'll compare our default `gemini-3.1-flash-lite` (lighter/cheaper) against `gemini-3.5-flash` (fuller model) on the same prompt — confirm both names appear in the list printed above for your key/region before running the next cell.


In [ ]:
import time

prompt = "Explain the difference between Agentic AI and RAG in 2 sentences."

for model_name in ["gemini-3.1-flash-lite", "gemini-3.5-flash"]:
    fast_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.3)
    start = time.time()
    result = fast_llm.invoke(prompt)
    elapsed = time.time() - start
    print(f"--- {model_name} ({elapsed:.1f}s) ---")
    print(get_text(result))
    print()


--- gemini-3.1-flash-lite (0.8s) ---
RAG (Retrieval-Augmented Generation) is a technique that provides an AI with external data to improve the accuracy of its responses to specific queries. In contrast, Agentic AI refers to autonomous systems capable of reasoning, planning, and using tools to execute multi-step tasks to achieve a goal.

--- gemini-3.5-flash (11.8s) ---
**RAG (Retrieval-Augmented Generation)** is a technique that improves an AI's responses by fetching relevant information from an external database to answer a specific query. In contrast, **Agentic AI** refers to autonomous systems that can proactively plan, make decisions, use various tools, and execute multi-step workflows to achieve complex goals.



**What to notice:** `gemini-3.1-flash-lite` usually answers noticeably faster, while `gemini-3.5-flash` tends to give a more thorough/nuanced answer — that speed-vs-capability tradeoff is the main thing you're choosing when you pick a model. In production you'd pick based on your actual requirements (latency budget, cost budget, task difficulty) — not just "use the best model available."

> If either model name above errors out for your key/region, re-run the previous cell and copy an exact name from the printed list.


## 4. Build a Basic Chatbot

The simplest possible "chatbot" is just:

```
User → LLM → Response
```

No memory, no history — every message is a fresh, independent call. Let's build that and immediately see its limitation.


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.7)

def chat(user_message):
    response = llm.invoke(user_message)
    return get_text(response)

print(chat("Hi! My name is Alex."))


Hi Alex! It's great to meet you. How are you doing today? Is there anything I can help you with?


In [ ]:
print(chat("What is my name?"))


I don’t know your name. As an AI, I don’t have access to your personal identity, documents, or private information unless you have explicitly shared it with me in this specific conversation.


**Expected behavior:** the model has no idea what your name is — it will say it doesn't know, or ask you to tell it. This isn't a bug. `chat()` calls `llm.invoke(...)` fresh each time, with *only* the new message. The previous exchange was never sent along, so as far as Gemini is concerned, this second call is the start of a brand-new conversation.

**This is true of essentially all LLM APIs**, not just Gemini: the model itself is stateless. Any "memory" you experience in a chat product (like the Gemini or ChatGPT web apps) is the *application* resending the conversation so far — which is exactly what we'll build next.


## 5. Conversation History & Memory

To make a chatbot feel continuous, the application has to keep track of what's been said and **resend it** on every call. Let's define a few terms precisely, because they get used loosely in practice:

- **Chat history** — the literal ordered list of messages exchanged so far (`Human: ...`, `AI: ...`, ...). This is the raw transcript.
- **Session** — one logical, continuous conversation instance (often identified by a session/conversation ID) that a chat history belongs to. A user might have many sessions over time.
- **Short-term memory** — the (usually recent) chat history that's actively resent to the model so it has context for the *current* session. This is what we're about to build.
- **Long-term memory** — information deliberately extracted and *persisted beyond a single session* (e.g. saved to a database or vector store) so it can be recalled in a future, separate session — "remembers you're vegetarian" three weeks later, not just three messages ago.

Today we're building **short-term memory** — a simple in-memory list of messages, kept only for the life of this notebook session.

### 5.1 LangChain's message types

LangChain represents a conversation as a list of typed messages:
- `SystemMessage` — instructions for how the model should behave.
- `HumanMessage` — something the user said.
- `AIMessage` — something the model said.


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = [SystemMessage(content="You are a friendly, concise assistant.")]

def chat_with_history(user_message):
    history.append(HumanMessage(content=user_message))
    response = llm.invoke(history)          # send the WHOLE conversation so far
    reply_text = get_text(response)
    history.append(AIMessage(content=reply_text))
    return reply_text

print(chat_with_history("My name is Alex."))


Hi Alex! It's nice to meet you. How can I help you today?


In [ ]:
print(chat_with_history("What is my name?"))


Your name is Alex!


**Expected behavior:** this time the model correctly answers "Alex" — because the full `history` list (system prompt + every human/AI turn so far) is sent to the model on *every single call*. Nothing is stored on Google's servers between calls; **we** are resending the transcript each time, and the model re-reads it from scratch.

Run the cell below to see exactly what's being sent under the hood:


In [ ]:
for msg in history:
    print(f"[{msg.type}] {msg.content}")


[system] You are a friendly, concise assistant.
[human] My name is Alex.
[ai] Hi Alex! It's nice to meet you. How can I help you today?
[human] What is my name?
[ai] Your name is Alex!


**A practical consequence:** the longer a conversation gets, the more text you resend (and pay for/wait on) with every single turn — this is why real chat products eventually summarize or trim older history instead of keeping it forever. We won't build that today, but it's worth knowing it's the next problem you'd hit.

**Recap of the four terms:**

| Term | What it is | Lifespan |
|---|---|---|
| Chat history | The raw list of messages | As long as you keep the list |
| Session | The conversation instance the history belongs to | Usually one user visit/interaction |
| Short-term (working) memory | History actively resent to the model for context | Current session only |
| Long-term memory | Facts deliberately saved for future sessions | Persists across sessions (needs a DB/vector store) |


## 6. Introduce RAG (Retrieval-Augmented Generation)

**Why not just paste the whole document into the prompt?**
- LLMs have a limited **context window** — very large or many documents simply won't fit.
- Even when it fits, sending a huge document on *every* question is slow and expensive (you pay/wait per token, every single call).
- Irrelevant surrounding text can distract the model and increase the chance of a wrong or hallucinated answer — needle-in-a-haystack problems are real.
- Real knowledge bases (wikis, ticket histories, codebases) are usually far bigger than any context window anyway.

**The idea behind RAG:** instead of sending the whole document, *search it first* for the few pieces that are actually relevant to the current question, and send only those pieces to the LLM.

```
Document
   ↓
Load
   ↓
Chunk
   ↓
Embedding
   ↓
Vector Database
   ↓
Similarity Search
   ↓
Relevant Chunks
   ↓
LLM
   ↓
Answer
```

**What each step means:**
- **Document** — your raw source of knowledge (a PDF, wiki page, text file, ...).
- **Load** — read it into a standard in-memory representation the framework understands.
- **Chunk** — split it into smaller, overlapping pieces. Smaller pieces mean each one is about *one specific topic*, so a search against them is more precise than searching whole documents.
- **Embedding** — convert each chunk of text into a vector (a list of numbers) that captures its *meaning*. Texts with similar meaning end up with similar vectors, even if they don't share exact words.
- **Vector database** — a store optimized for holding these vectors and quickly finding the ones closest to a given query vector.
- **Similarity search** — embed the user's *question* the same way, then find the chunks whose vectors are closest to it — i.e. the most semantically relevant chunks.
- **Relevant chunks** — the small handful of chunks that actually matter for this question (out of possibly thousands).
- **LLM** — given the question *plus* those relevant chunks as context, generate an answer grounded in the real document.
- **Answer** — the final response, ideally only using facts that were actually in the retrieved chunks.

Let's build this end-to-end with a tiny sample document.


## 7. Build a Small RAG System

To keep things self-contained, we'll use a small fictional company handbook as our "document" — no file upload needed. Notice it contains specific facts we can later ask questions about, and also leaves plenty of things *unanswered* on purpose.


In [ ]:
sample_document = """
Acme Robotics — Employee Handbook (Excerpt)

Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays may be worked remotely, subject to manager
approval. Fully remote arrangements require VP-level sign-off and are reviewed quarterly.

Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

Expense Reimbursement:
Employees may claim reimbursement for approved business expenses (travel, client meals,
conference fees) by submitting receipts within 30 days of the expense. Reimbursements
are processed within 10 business days of approval. Personal expenses, including gym
memberships and home internet, are not reimbursable.

Equipment Policy:
New employees receive a laptop and a one-time $200 home-office setup stipend during
their first 90 days. Equipment remains the property of Acme Robotics and must be
returned upon termination of employment.
"""

print(sample_document[:200], "...")



Acme Robotics — Employee Handbook (Excerpt)

Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays ...


### 7.1 Document Loader

In a real project you'd use a loader like `PyPDFLoader` or `TextLoader` to read a file from disk. Since our text already lives in the notebook, we just wrap it in LangChain's `Document` object — the same shape every loader would produce.


In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=sample_document, metadata={"source": "acme_handbook.txt"})]
print(f"Loaded {len(docs)} document(s), {len(docs[0].page_content)} characters.")


Loaded 1 document(s), 1188 characters.


### 7.2 Text Splitter (chunking)

`RecursiveCharacterTextSplitter` tries to split on natural boundaries (paragraphs, then sentences, then words) so chunks stay coherent. `chunk_overlap` repeats a little text between consecutive chunks so we don't lose context right at a chunk boundary.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{c.page_content}\n")


Split into 6 chunks.

--- Chunk 1 ---
Acme Robotics — Employee Handbook (Excerpt)

--- Chunk 2 ---
Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays may be worked remotely, subject to manager
approval. Fully remote arrangements require VP-level sign-off and are reviewed quarterly.

--- Chunk 3 ---
Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

--- Chunk 4 ---
Expense Reimbursement:
Employees may claim reimbursement for approved business expenses (travel, client meals,
conference fees) by submitting receipts within 30 days of the expense. Reimbursements
are processed within 10 business days of approval. Personal expenses, including gym

--- 

### 7.3 Embeddings

Now we turn each chunk into a vector using Gemini's embedding model.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

embedding_model_name = None
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        embedding_model_name = m.name
        break

print("Using embedding model:", embedding_model_name)

embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model_name)

# Sanity check: embed one chunk and look at the vector's shape
sample_vector = embeddings.embed_query(chunks[0].page_content)
print(f"Embedding vector length: {len(sample_vector)}")
print("First 8 numbers:", sample_vector[:8])


Using embedding model: models/gemini-embedding-001
Embedding vector length: 3072
First 8 numbers: [-0.010842091, 0.03319727, -0.0018307053, -0.047145642, -0.00018826101, 0.013176389, 0.026611274, -0.003846929]


**What you're looking at:** that's the "meaning" of the first chunk expressed as ~a few hundred numbers. It means nothing to a human, but two chunks about similar topics will have vectors that are mathematically close together — that's the entire trick similarity search relies on.

### 7.4 Vector Store

For a workshop, we want zero infrastructure — no server to run, nothing to configure. LangChain's `InMemoryVectorStore` keeps everything in RAM for the life of this runtime, which is perfect here (in a real production system you'd use something persistent like Chroma, FAISS, or a managed vector DB).


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

print(f"Vector store now holds {len(chunks)} embedded chunks.")


Vector store now holds 6 embedded chunks.


### 7.5 Retriever + similarity search

A **retriever** wraps "embed the query, then search the vector store" into one call. Let's try it directly before wiring it up to the LLM, so you can see exactly what gets retrieved.


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})  # return the top-2 most relevant chunks

results = retriever.invoke("How much vacation time do I get?")
for i, doc in enumerate(results):
    print(f"--- Retrieved chunk {i+1} ---\n{doc.page_content}\n")


--- Retrieved chunk 1 ---
Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

--- Retrieved chunk 2 ---
Equipment Policy:
New employees receive a laptop and a one-time $200 home-office setup stipend during
their first 90 days. Equipment remains the property of Acme Robotics and must be
returned upon termination of employment.



**Expected behavior:** the leave-policy chunk should come back as the top match, even though the question ("vacation time") doesn't use the exact same words as the document ("paid vacation")  — this is the embedding capturing *meaning*, not just keyword matching.

### 7.6 Wire it up to Gemini — a Q&A function

We give the LLM an explicit instruction: **answer only from the provided context, and say so if the answer isn't there.** This is the single most important line in a RAG system — without it, the model will happily guess.


In [ ]:
RAG_PROMPT = """Answer the question using ONLY the context below.
If the answer is not contained in the context, respond exactly with:
"I don't have that information in the document."
Do not use outside knowledge and do not guess.

Context:
{context}

Question: {question}

Answer:"""

def ask(question, k=2):
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    return get_text(llm.invoke(prompt))

print(ask("How many vacation days do full-time employees get?"))


Full-time employees accrue 18 days of paid vacation per year.


Now let's try a question whose answer **is not** in the document at all:


In [ ]:
print(ask("What is the capital of France?"))


I don't have that information in the document.


In [ ]:
print(ask("Does Acme Robotics offer a 401k match?"))


I don't have that information in the document.


**Expected behavior:** the first `ask()` call should correctly answer "18 days" (pulled straight from the Leave Policy chunk). The other two should both come back with the "I don't have that information in the document" fallback — even though the model *could* answer the France question from its own general knowledge, we explicitly told it not to use outside knowledge, so it stays grounded in *only* what we retrieved. That's the difference between a RAG system and "just asking an LLM."


## 8. Final Mini AI System — Putting It All Together

Let's combine everything from today into one small application:

```
User
 ↓
Chat Interface
 ↓
Conversation History   (Section 5)
 ↓
Retriever              (Section 7)
 ↓
Relevant Context       (Section 7)
 ↓
Gemini                 (Section 2/3)
 ↓
Response
```

On every turn we (1) retrieve context relevant to the *new* question, (2) combine it with the conversation history so far, (3) send everything to Gemini, and (4) record the new turn into history for next time.


In [ ]:
conversation_history = []  # list of HumanMessage / AIMessage, growing each turn

def rag_chat(user_message):
    # 1. Retrieve document context relevant to THIS message
    relevant_docs = retriever.invoke(user_message)
    context = "\n\n".join(doc.page_content for doc in relevant_docs)

    # 2. Build fresh instructions (including retrieved context) for this turn
    system_message = SystemMessage(content=(
        "You are a helpful assistant for Acme Robotics employees. "
        "Use the CONTEXT below if it's relevant to the question. "
        "If the answer isn't in the context AND isn't something already established "
        "earlier in this conversation, say you don't have that information. "
        "Do not make up policy details.\n\nCONTEXT:\n" + context
    ))

    # 3. Combine: system instructions + everything said so far + the new message
    messages = [system_message] + conversation_history + [HumanMessage(content=user_message)]
    response = llm.invoke(messages)
    reply_text = get_text(response)

    # 4. Record this turn so future turns remember it
    conversation_history.append(HumanMessage(content=user_message))
    conversation_history.append(AIMessage(content=reply_text))

    return reply_text

print(rag_chat("Hi, my name is Alex."))


Hi Alex! How can I help you today?


In [ ]:
print(rag_chat("How many sick days do I get per year?"))


Hi Alex, as a full-time employee at Acme Robotics, you get 10 paid sick days per year.


In [ ]:
print(rag_chat("And what's my name again?"))


Your name is Alex.


**Expected behavior:**
- Turn 1 is just a friendly greeting (no document context needed, but memory now has "the user's name is Alex").
- Turn 2 pulls the sick-day fact straight from the retrieved handbook chunk, exactly like Section 7.
- Turn 3 is answered correctly ("Alex") *not* from the document, but from `conversation_history` — proving both memory and retrieval are working together in the same system.

## Recap: what you actually built today

```
LLM  →  Parameters  →  Chatbot  →  History  →  Memory  →  Embeddings  →  Vector DB  →  Retrieval  →  RAG  →  AI Application
```

You started with a single stateless call to Gemini, and ended with an application that:
- remembers the conversation so far (short-term memory),
- looks up facts from a real document instead of relying on the model's own knowledge (RAG),
- refuses to answer when it genuinely doesn't know (grounded, not hallucinating),
- and combines all of that into one coherent multi-turn experience.

That's already meaningfully more than "call an LLM API" — it's a small, real **AI system**, with the same shape (interface → memory → retrieval → LLM → response) that production AI products use, just at a smaller scale.

**Natural next steps** (not covered today, but now that these fundamentals are solid, worth knowing exist):
- **Tools / function calling & agents** — letting the LLM call your own code/APIs to take actions, not just answer questions.
- **Persistent, long-term memory** — saving key facts to a real database so they survive across sessions, not just within one run.
- **Production-grade vector stores** — Chroma, FAISS, pgvector, or managed services, instead of the in-memory store we used today.
- **Evaluation & guardrails** — systematically testing whether your AI system's answers are accurate, safe, and grounded before shipping it.
- **Streaming responses** — sending tokens to the user as they're generated instead of waiting for the full answer.

Great work — you now have a working mental model *and* working code for how a real GenAI application is built, one layer at a time.
